In [66]:

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

In [67]:
file_path=r'D:\Saran\RAG\CSR MODULES (1).pdf'

loader=PyPDFLoader(file_path)

documents=loader.load()



Ignoring wrong pointing object 1006 0 (offset 0)
Ignoring wrong pointing object 7232 0 (offset 0)
Ignoring wrong pointing object 7239 0 (offset 0)
Ignoring wrong pointing object 7240 0 (offset 0)


In [68]:
for doc in documents:
    print(doc)

page_content='1 
 
 TABLE OF CONTENTS  
 
CSR 1 MODULE..................................................................................................................................................................... 6 
Cha pter One: Overview......................................................................................................................................................... 7 
What to Expect in Part 1 & 2 ................................................................................................................................................. 8 
Who Are Our Customers? .................................................................................................................................................... 9 
What Makes Some one  a Customer? .................................................................................................................................10 
Sunshine  Law a nd Red Fla g Rule .............................................

In [69]:
import pytesseract
from PIL import Image
import fitz

In [70]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [71]:
# doc=fitz.open(file_path)

# text=""

# for page in doc:
#     pix=page.get_pixmap()
#     img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
#     text += pytesseract.image_to_string(img)
    

# print(text)
import fitz
import pytesseract
from PIL import Image

doc = fitz.open("CSR MODULES (1).pdf")

documents = []

for i, page in enumerate(doc):

    # 1️⃣ Extract normal text
    text = page.get_text()

    # 2️⃣ Convert full page to image (for OCR)
    pix = page.get_pixmap(matrix=fitz.Matrix(3, 3))
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

    # 3️⃣ OCR on image
    ocr_text = pytesseract.image_to_string(img)

    # 4️⃣ Combine both
    full_text = text + "\n" + ocr_text

    documents.append({
        "page": i + 1,
        "text": full_text
    })

# Print output
for doc in documents:
    print(doc)

{'page': 1, 'text': '1 \n \n TABLE OF CONTENTS \n \nCSR 1 MODULE..................................................................................................................................................................... 6 \nChapter One: Overview......................................................................................................................................................... 7 \nWhat to Expect in Part 1 & 2 ................................................................................................................................................. 8 \nWho Are Our Customers? .................................................................................................................................................... 9 \nWhat Makes Someone a Customer? .................................................................................................................................10 \nSunshine Law and Red Flag Rule ....................................

In [77]:
def clean_text(text):
    text=text.replace("/n"," ")
    text=" ".join(text.split())
    return text

cleaned_docs = [
    Document(
        page_content=clean_text(doc["text"]),
        metadata=doc.get("metadata", {})
    )
    for doc in documents
]

In [78]:
from langchain_text_splitters import RecursiveCharacterTextSplitter



In [84]:
text_splitter=RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ".", " "],
    chunk_size=2000,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(cleaned_docs)

print("Total chunks:", len(chunks))
print("Sample chunk:", chunks[0].page_content[:300])

Total chunks: 359
Sample chunk: 1 TABLE OF CONTENTS CSR 1 MODULE..................................................................................................................................................................... 6 Chapter One: Overview...............................................................................


In [85]:
from sentence_transformers import SentenceTransformer



In [86]:
model=SentenceTransformer("all-MiniLM-L6-v2")

texts=[doc.page_content for doc in chunks]

embeddings=model.encode(texts)

print(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1838.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[-0.03220081  0.02811461 -0.09728415 ...  0.08040222 -0.00749126
  -0.00160867]
 [ 0.01869667  0.10104986 -0.01926927 ...  0.0486389  -0.05707153
  -0.0104619 ]
 [ 0.03296016 -0.05019557 -0.07104542 ... -0.02123609 -0.0289512
  -0.04188145]
 ...
 [-0.08182513 -0.06270869 -0.02304456 ...  0.07877127 -0.05122925
  -0.02978859]
 [-0.08048151 -0.01037387 -0.0047481  ...  0.07350323 -0.01357979
   0.00019854]
 [-0.03602268 -0.01678916  0.04543771 ...  0.04121661 -0.11144886
   0.01096648]]


In [87]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [88]:
embeddings=HuggingFaceEmbeddings()
texts=[doc.page_content for doc in chunks]

db=FAISS.from_texts(texts,embeddings)
query="what is Red Flag Rule"

docs=db.similarity_search(query)

for doc in docs:
    print(doc)

C:\Users\AIT\AppData\Local\Temp\ipykernel_4496\165090109.py:1: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings=HuggingFaceEmbeddings()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2948.54it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


page_content='11 Sunshine Law and Red Flag Rule Sunshine Law: Florida's Government in the Sunshine Law, commonly called the Sunshine Law, passed in 1967. It requires that all meetings of any state, county, or municipal board or commission in Florida be open to the public and declares that actions taken at closed meetings are not binding. These laws aim to promote transparency and accountability in government operations, and to ensure that the actions and decisions of public officials are conducted openly and are accessible to the public they serve. Additionally, they may require that government records and documents be available for public inspection and copying, with certain exemptions for sensitive information like personal privacy or national security. Red Flag Rule: The "Red Flag Rule" refers to a specific law aimed at combating identity theft. It requires certain businesses and organizations to implement identity theft prevention programs designed to detect, prevent, and mitigate 

In [90]:
retriever=db.as_retriever()

query='CREATING A PROFILE IN CCS' 

docs=retriever.invoke(query)

for doc in docs:
    print(doc.page_content)

50 How to Create Profile How to Create Profile CREATING A PROFILE IN CCS YB Control Centra Search x + € > CSC & utilities-cloud oracteindustry com/<7B6p1/prod/ccs/web/cis jsp CG Managed bookmaits €B COIN I Caco Waning @ Cowen QU CR Tips ‘© Customer Cloud Service QqaoFzae 9 91a Control Central Search > Fwortte Lines Todos Pay Pans Ortire Unitty Exchange we rete ee maven Poot Contmcrfewere Person Cormac Toe Contac termaten Eero Format No ooe Acero toute . —— _ winwreg Fecretrs +B Prone Cot Pree an] (om) SD can + © tw rear tet ~" haernetvecom ones Senne seectare 320 Pert SSSEBEUES +28 een ae Work List ow BA 50
82 On the search page of CCS, search the desired authorized user by first and last name to ensure they don’t already have an account or profile. Once you type in LAST NAME,FIRST NAME press enter. If nothing comes up, ask them what their MIDDLE INITIAL is, add it to the search, and press enter again. If nothing comes up, they don’t have account or profile, so create a new account for